In [1]:
import pandas as pd
import numpy as np

In [2]:
pl = pd.read_csv("Data/Context/Poland.csv")

In [3]:
pl = pl.drop(["Unnamed: 0"], axis=1)

In [4]:
pl_2007_2013 = pl.loc[7:13]

In [5]:
pl_2007_2013

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,WTDA,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
7,2007,0.41,0.36,0.28,0.77,0.88,42.0,0.957,0.959,48.6,...,3.93,NaN,11.0,45.0,58.940544,50.164067,60.0,51.654726,52.749451,NaN
8,2008,0.55,0.46,0.45,0.84,0.95,46.0,0.979,0.963,53.1,...,3.69,NaN,11.0,45.0,59.448350,54.284358,60.0,51.550416,56.847812,NaN
9,2009,0.65,0.52,0.44,0.98,1.03,50.0,0.979,0.963,59.0,...,3.88,NaN,11.0,45.0,61.376662,54.635953,60.0,50.682655,56.847812,NaN
10,2010,0.71,0.63,0.51,1.02,1.04,53.0,0.978,0.964,62.3,...,4.00,65.079783,7.0,39.0,78.465630,54.834479,60.0,53.323655,56.847812,NaN
11,2011,0.79,0.60,0.56,0.96,1.03,55.0,0.978,0.964,62.0,...,3.85,67.008433,7.0,39.0,78.541563,54.910657,60.0,63.901225,56.847812,NaN
12,2012,0.82,0.66,0.66,1.00,1.06,58.0,0.978,0.964,62.3,...,3.72,68.038453,7.0,39.0,78.623297,66.039726,60.0,65.669944,56.847812,NaN
13,2013,0.84,0.66,0.65,1.05,1.00,60.0,0.975,0.961,62.8,...,3.77,72.786018,7.0,39.0,79.040430,70.275720,60.0,72.179181,64.976227,NaN


In [6]:
def missing_values(df):
    for feature in df.columns:
        missing_values = df[feature].isna().sum()
        percentage = missing_values/len(df) * 100
        print(f'{feature} - {missing_values} - {round(percentage, 1)} %')

In [7]:
missing_values(pl_2007_2013)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 4 - 57.1 %
PR - 0 - 0.0 %
DI - 2 - 28.6 %
SOARS - 0 - 0.0 %
TOGP - 0 - 0.0 %
IPAB - 3 - 42.9 %
PTIP - 0 - 0.0 %
QOES - 0 - 0.0 %
AOLT - 0 - 0.0 %
FLTA - 0 - 0.0 %
EBOF - 0 - 0.0 %
FOWD - 0 - 0.0 %
QOMS - 0 - 0.0 %
CLER - 0 - 0.0 %
WTDA - 0 - 0.0 %
EDBS - 3 - 42.9 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 7 - 100.0 %


In [8]:
mode = pl_2007_2013['DI'].mode()[0]
pl_2007_2013['DI'].fillna(value=mode, inplace=True)

/var/folders/41/w_0mz3gn2vn0w0dz8wywr8_r0000gn/T/ipykernel_34581/2690129465.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pl_2007_2013['DI'].fillna(value=mode, inplace=True)


In [9]:
pl_2007_2013 = pl_2007_2013.drop(["ELFE", "CFI", "IPAB", "EDBS"], axis=1)

In [10]:
def fill_with_mean(df):
    for feature in df.columns:
        mean = df[feature].mean()
        df[feature].fillna(value=mean, inplace=True)

In [11]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [12]:
def handle_outliers(df, continuous_features):

    print("Original Shape:", df.shape)
    
    outlier_summary = {}
    
    for feature in continuous_features:
        if feature in df.columns:
            Q1 = df[feature].quantile(0.25)
            Q3 = df[feature].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_mask = (df[feature] < lower_bound) | (df[feature] > upper_bound)
            num_outliers = outlier_mask.sum()
            
            if num_outliers > 0:
                non_outlier_mean = df.loc[~outlier_mask, feature].mean()
                
                df.loc[outlier_mask, feature] = non_outlier_mean
                
                print(f"{feature}: Found {num_outliers} outliers, replaced with mean = {non_outlier_mean:.2f}")
            else:
                print(f"{feature}: No outliers found")
    
    print("\nFinal Shape:", df.shape)
    print(f"Total features processed: {len(continuous_features)}")
    
    return df

In [13]:
pl_2007_2013_no_o = handle_outliers(pl_2007_2013, continuous_data)

Original Shape: (7, 30)
DBP: No outliers found
DBD: No outliers found
SSB: No outliers found
DWCP: Found 1 outliers, replaced with mean = 55.81
PT: No outliers found
EC: Found 2 outliers, replaced with mean = 56.85
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 30)
Total features processed: 9


In [14]:
pl_2007_2013_no_o

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,QOMS,CLER,WTDA,DBP,DBD,SSB,DWCP,PMSI,PT,EC
7,2007,0.41,0.36,0.28,0.77,0.88,42.0,0.957,0.959,48.6,...,4.41,4.00,3.93,11.0,45.0,58.940544,50.164067,60.0,51.654726,56.847812
8,2008,0.55,0.46,0.45,0.84,0.95,46.0,0.979,0.963,53.1,...,4.51,3.86,3.69,11.0,45.0,59.448350,54.284358,60.0,51.550416,56.847812
9,2009,0.65,0.52,0.44,0.98,1.03,50.0,0.979,0.963,59.0,...,4.47,4.11,3.88,11.0,45.0,61.376662,54.635953,60.0,50.682655,56.847812
10,2010,0.71,0.63,0.51,1.02,1.04,53.0,0.978,0.964,62.3,...,4.23,4.10,4.00,7.0,39.0,78.465630,54.834479,60.0,53.323655,56.847812
11,2011,0.79,0.60,0.56,0.96,1.03,55.0,0.978,0.964,62.0,...,4.03,4.00,3.85,7.0,39.0,78.541563,54.910657,60.0,63.901225,56.847812
12,2012,0.82,0.66,0.66,1.00,1.06,58.0,0.978,0.964,62.3,...,3.97,4.10,3.72,7.0,39.0,78.623297,66.039726,60.0,65.669944,56.847812
13,2013,0.84,0.66,0.65,1.05,1.00,60.0,0.975,0.961,62.8,...,3.96,4.13,3.77,7.0,39.0,79.040430,55.811540,60.0,72.179181,56.847812


In [15]:
pl_2007_2013_no_o.to_csv('Data/Context/Poland_2007_2013.csv')

In [16]:
pl_2007_2013_no_o = pl_2007_2013_no_o.drop(["TIME_PERIOD"], axis=1)

In [17]:
pl_2007_2013_l = pl_2007_2013_no_o.loc[13]

In [18]:
pl_2007_2013_l.to_csv('Data/Context/Poland_2007_2013_last.csv')

In [19]:
pl_2014_2020 = pl.loc[14:20]

In [20]:

missing_values(pl_2014_2020)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 0 - 0.0 %
PR - 0 - 0.0 %
DI - 0 - 0.0 %
SOARS - 3 - 42.9 %
TOGP - 3 - 42.9 %
IPAB - 3 - 42.9 %
PTIP - 3 - 42.9 %
QOES - 3 - 42.9 %
AOLT - 3 - 42.9 %
FLTA - 3 - 42.9 %
EBOF - 3 - 42.9 %
FOWD - 3 - 42.9 %
QOMS - 3 - 42.9 %
CLER - 3 - 42.9 %
WTDA - 4 - 57.1 %
EDBS - 0 - 0.0 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 1 - 14.3 %


In [21]:
pl_2014_2020 = pl_2014_2020.drop(["SOARS", "TOGP", "IPAB", "PTIP", "QOES", "AOLT", "FLTA", "EBOF",
                                 "FOWD", "QOMS", "CLER", "WTDA"], axis=1)

In [22]:
fill_with_mean(pl_2014_2020)

In [23]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [24]:
pl_2014_2020_no_o = handle_outliers(pl_2014_2020, continuous_data)

Original Shape: (7, 22)
DBP: No outliers found
DBD: No outliers found
EDBS: No outliers found
SSB: No outliers found
DWCP: Found 1 outliers, replaced with mean = 77.17
PT: No outliers found
EC: No outliers found
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 22)
Total features processed: 9


In [25]:
pl_2014_2020_no_o.to_csv('Data/Context/Poland_2014_2020.csv')

In [26]:
pl_2014_2020_no_o = pl_2014_2020_no_o.drop(["TIME_PERIOD"], axis=1)

In [27]:
pl_2014_2020_no_o

,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,CFI,...,DI,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
14,0.87,0.77,0.68,1.06,1.11,61.0,0.975,0.957,66.6,40.6,...,7.47,73.637090,5.0,37.0,82.524812,77.165373,66.0,72.292209,64.826246,5.116667
15,0.80,0.74,0.69,0.95,1.04,62.0,0.965,0.874,68.0,40.2,...,7.09,74.112611,5.0,37.0,82.560572,75.499244,66.0,72.258398,63.437357,5.300000
16,0.59,0.65,0.76,0.91,0.84,62.0,0.745,0.783,73.3,40.2,...,6.83,76.888425,5.0,37.0,82.711080,75.532289,66.0,79.141239,63.437357,5.200000
17,0.40,0.55,0.70,0.81,0.78,60.0,0.709,0.690,76.0,42.0,...,6.67,77.649190,5.0,37.0,82.752840,77.966096,66.0,79.105836,63.437357,5.000000
18,0.38,0.55,0.62,0.87,0.71,60.0,0.709,0.595,77.5,41.7,...,6.67,77.825980,5.0,37.0,82.780110,77.975538,66.0,79.495816,63.437357,4.800000
19,0.39,0.51,0.61,1.01,0.67,58.0,0.704,0.572,80.4,41.3,...,6.62,76.901921,5.0,37.0,82.848012,77.999049,66.0,76.490761,64.363283,5.100000
20,0.52,0.32,0.62,0.85,0.62,56.0,0.707,0.570,83.2,40.0,...,6.85,76.389916,5.0,37.0,82.908585,78.020022,66.0,76.831746,64.363283,5.300000


In [28]:
pl_2014_2020_l = pl_2014_2020_no_o.loc[20]

In [29]:
pl_2014_2020_l

ROL      0.520000
GE       0.320000
COC      0.620000
RQ       0.850000
VAA      0.620000
CPI     56.000000
FOP      0.707000
JI       0.570000
IUI     83.200000
CFI     40.000000
PR      63.000000
DI       6.850000
EDBS    76.389916
DBP      5.000000
DBD     37.000000
SSB     82.908585
DWCP    78.020022
PMSI    66.000000
PT      76.831746
EC      64.363283
ELFE     5.300000
Name: 20, dtype: float64

In [30]:
pl_2014_2020_l.to_csv('Data/Context/Poland_2014_2020_last.csv')